# Replica staleness through Pgpool-II, and the 500s it is supposed to explain

**Question.** When a write goes through Pgpool on one connection, can a read on a *different*
connection fail to see it — and if so, does that produce the 500s in `local-k8s`
`.memory/active-issues.md` **#15**?

#15 was once labelled a "PG-HA read-after-write signature". That label was asserted and never
established. What the Polaris logs carry is a `NullPointerException` from
`IcebergExceptionMapper`, and the trigger looks like a **fresh parent entity, not load**: 3 of 3
brand-new catalogs 500 on the first namespace created in them.

**Why the mechanism is possible at all**, read out of the chart rather than assumed:

| link | setting | value |
|---|---|---|
| replication is asynchronous | `postgresql.syncReplication` (never set) | `false` |
| reads are load-balanced onto standbys | `pgpool.useLoadBalancing` (never set) | `true` |
| the read backend is fixed **per session** | `statement_level_load_balance` | `off` |
| write-pinning protects only **within** one session | `disableLoadBalancingOnWrite` | `always` |
| **replication lag is never checked** | `delay_threshold` — set nowhere | pgpool default `0` |

Polaris writes on one Agroal connection and resolves on another — a different Pgpool session with
no write of its own, load-balanced, and with no `delay_threshold` free to land on a standby that
is arbitrarily far behind.

**Method.** Pause WAL replay on every standby so staleness is guaranteed rather than hoped for,
then measure two things while it is paused:

1. **§7 — does a pooled read miss a committed write?** (the mechanism)
2. **§8 — does the #15 catalog→namespace ladder 500?** (the symptom), re-run in §10 against
   healthy replication as a control

## ⚠ Read before running

- **Local only.** This pauses replication *and* creates Polaris catalogs. `require_not_prod()`
  blocks `prod`; the extra assertion also blocks `dev`, which is shared.
- **Keep the paused window to minutes.** §4 prints which WAL-retention risk applies.
- **If anything goes wrong**, run the PANIC cell — idempotent, works standalone.
- Needs `kubectl` with the context on `orbstack`.

## Correction — run 1 (2026-09-17) measured its own instrument

The first version detected routing with `pg_is_in_recovery()` *inside the measured query*. All
12 sessions reported `primary` and the notebook printed REFUTED. That verdict was not safe:
`pg_is_in_recovery()` is **VOLATILE**, and Pgpool decides read-vs-write by parsing the statement,
so the detector may have been what forced every read to the primary. Nothing proved the
connection even traversed Pgpool.

This version fixes both: **§2b refuses to continue unless `SHOW pool_nodes` answers**, and
routing is measured from Pgpool's own `select_cnt` counters around a plain query, which cannot
perturb the decision it is measuring. §7b keeps the old instrumented query beside it, purely to
show the disagreement if there is one.

## 1 — Setup and guards

In [ ]:
import sys, pathlib

_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / "src"))
from nb_support import *  # noqa: F401,F403

init_env("local")
require_not_prod("pause WAL replay and create catalogs")

# require_not_prod only blocks `prod`. `dev` is a SHARED company cluster: pausing
# replay there would degrade other people's reads. Local-only, no override.
assert ENV == "local", (
    f"Refusing to run against env={ENV!r}. This notebook pauses WAL replay and "
    "is local-only by design."
)

print(f"env      = {ENV}")
print(f"postgres = {PG_HOST}:{PG_PORT}/{PG_DB} as {PG_USER}")
print(f"polaris  = {POLARIS_URL}")

In [ ]:
import subprocess
import time

import pandas as pd
import psycopg2

NAMESPACE = "datahub-hynix"
STS = "benchmarks-postgresql-postgresql-ha-postgresql"
PGPOOL_SVC = "benchmarks-postgresql-postgresql-ha-pgpool"
CONTAINER = "postgresql"
PW_FILE = "/opt/bitnami/postgresql/secrets/postgres-password"
REPLICAS = 3
PROBE_TABLE = "rw_probe"
N_SESSIONS = 12
N_RUNGS = 3          # #15's shape: 3 brand-new catalogs, first namespace in each
EXPECT_CTX = "orbstack"

ctx = subprocess.run(
    ["kubectl", "config", "current-context"], capture_output=True, text=True
).stdout.strip()
assert ctx == EXPECT_CTX, f"kubectl context is {ctx!r}, expected {EXPECT_CTX!r}"
print(f"kubectl context = {ctx}")

## 2 — Helpers

In [ ]:
def kexec(ordinal, sql, db=None):
    """Run SQL as superuser directly on one Postgres pod, bypassing Pgpool.

    The password is read from the file the chart already mounts inside the pod,
    so no credential appears anywhere in this notebook.

    Connects over TCP to 127.0.0.1 rather than the Unix socket, because
    `inet_server_addr()` returns NULL on a socket connection -- which is why the
    role table in run 1 had an empty `addr` column.
    """
    db = db or PG_DB
    inner = (
        f"PGPASSWORD=$(cat {PW_FILE}) "
        f'psql -h 127.0.0.1 -U postgres -d {db} -tAc "$0"'
    )
    r = subprocess.run(
        [
            "kubectl", "-n", NAMESPACE, "exec", f"{STS}-{ordinal}",
            "-c", CONTAINER, "--", "sh", "-c", inner, sql,
        ],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"pg-{ordinal}: {r.stderr.strip()}")
    return r.stdout.strip()


def pool_sql(sql, fetch=True):
    """Run SQL through Pgpool on a BRAND NEW connection, then close it.

    Deliberately unpooled, and a deliberate exception to this repo's
    close-the-pool convention: there is no pool here to close. Because
    `statement_level_load_balance = off`, Pgpool chooses the read backend once
    per session and keeps it, so reusing a connection would ask the same backend
    every time and measure nothing.

    One call == one session == one independent routing decision, which is the
    shape Polaris produces: an Agroal pool of 10-300 connections, each its own
    Pgpool session.
    """
    conn = psycopg2.connect(**PG_CONFIG)
    try:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall() if fetch else None
    finally:
        conn.close()


#: `SHOW pool_nodes` column order. The tail varies by Pgpool version, so extra
#: columns are named positionally rather than assumed away.
PN_COLS = [
    "node_id", "hostname", "port", "status", "pg_status", "lb_weight", "role",
    "pg_role", "select_cnt", "load_balance_node", "replication_delay",
    "replication_state", "replication_sync_state", "last_status_change",
]


def pool_nodes_df():
    rows = pool_sql("SHOW pool_nodes;")
    n = len(rows[0])
    cols = PN_COLS[:n] + [f"col{i}" for i in range(len(PN_COLS), n)]
    return pd.DataFrame(rows, columns=cols)


def select_counts():
    """Pgpool's own per-backend SELECT counter. Measuring routing from here
    instead of from a function inside the query is the whole correction: this
    cannot influence the decision it is reporting."""
    df = pool_nodes_df()
    return {r.hostname: int(r.select_cnt) for r in df.itertuples()}


print("helpers ready")

## 2b — Prove we are actually talking to Pgpool

Run 1 never checked this. If `PG_HOST` reaches a Postgres backend directly, every read lands on
that one node and the output is **indistinguishable** from "Pgpool does not load-balance".
`SHOW pool_nodes` answers only on a Pgpool connection.

In [ ]:
try:
    nodes = pool_nodes_df()
except Exception as e:  # noqa: BLE001 - the message is the point
    raise AssertionError(
        f"{PG_HOST}:{PG_PORT} does not answer SHOW pool_nodes, so it is NOT Pgpool. "
        f"The routing question cannot be asked from here. ({e})"
    )

display(nodes[["node_id", "hostname", "status", "lb_weight", "role", "select_cnt",
               "load_balance_node"]])

if (nodes.status.astype(str).str.lower() != "up").any():
    print("\nWARNING: not every backend is 'up' - routing results will be skewed.")
if (pd.to_numeric(nodes.lb_weight, errors="coerce").fillna(0) == 0).any():
    print("\nWARNING: at least one backend has lb_weight 0 - it will never serve a read,")
    print("         which alone would explain a 'no balancing' result.")
print(f"\nconfirmed: {PG_HOST}:{PG_PORT} is Pgpool, {len(nodes)} backends")

## 🚨 PANIC / RECOVERY

**Run this cell on its own if anything goes wrong**, including after a kernel restart (run §1 and
§2 first). Idempotent: resumes replay on every pod in recovery, leaves the primary alone.

In [ ]:
def resume_all(verbose=True):
    """Resume WAL replay on every standby. Safe to call any number of times."""
    out = []
    for i in range(REPLICAS):
        try:
            if kexec(i, "SELECT pg_is_in_recovery();") != "t":
                out.append((i, "primary - nothing to do"))
                continue
            kexec(i, "SELECT pg_wal_replay_resume();")
            out.append((i, kexec(i, "SELECT pg_get_wal_replay_pause_state();")))
        except Exception as e:  # noqa: BLE001 - recovery path must not raise
            out.append((i, f"ERROR: {e}"))
    if verbose:
        for i, s in out:
            print(f"pg-{i}: {s}")
    return out


# resume_all()   # <- uncomment and run THIS CELL ONLY if a run was interrupted

## 3 — Who is the primary?

Derived, never assumed. Measured 2026-09-17: the primary is **pg-1**. A hand-written script that
hardcoded `for i in 1 2` tried to pause the primary and left a standby running, silently
invalidating its own run.

In [ ]:
rows = []
for i in range(REPLICAS):
    in_recovery = kexec(i, "SELECT pg_is_in_recovery();") == "t"
    rows.append(
        {
            "ordinal": i,
            "pod": f"{STS}-{i}",
            "role": "standby" if in_recovery else "primary",
            "addr": kexec(i, "SELECT inet_server_addr();"),
        }
    )

roles = pd.DataFrame(rows)
display(roles)

primaries = roles.loc[roles.role == "primary", "ordinal"].tolist()
STANDBYS = roles.loc[roles.role == "standby", "ordinal"].tolist()
assert len(primaries) == 1, f"expected exactly one primary, found {primaries}"
assert STANDBYS, "no standbys found - nothing to make stale"
PRIMARY = primaries[0]

print(f"primary = pg-{PRIMARY}   standbys = {STANDBYS}")

## 4 — Pre-flight: which WAL-retention risk are you taking?

Pausing stops *replay*, not *receipt*. WAL keeps arriving; what that costs depends on slots.

In [ ]:
slots = int(kexec(PRIMARY, "SELECT count(*) FROM pg_replication_slots;"))
wal_keep = kexec(PRIMARY, "SHOW wal_keep_size;")

print(f"replication slots on primary : {slots}")
print(f"wal_keep_size                : {wal_keep}\n")

if slots > 0:
    print("  SLOTS PRESENT -> the primary retains WAL for a paused standby WITHOUT BOUND.")
    print("  Watch the filesystem below. Keep the paused window to minutes.")
else:
    print(f"  NO SLOTS -> retention is capped at {wal_keep}. A paused standby that falls")
    print("  further behind than that CANNOT catch up on resume and needs a full re-clone")
    print("  (`usePgRewind` is false on this chart, so pg_basebackup, not a rewind).")

print("\nfilesystem on the primary:")
print(
    subprocess.run(
        ["kubectl", "-n", NAMESPACE, "exec", f"{STS}-{PRIMARY}", "-c", CONTAINER,
         "--", "df", "-h", "/bitnami/postgresql"],
        capture_output=True,
        text=True,
    ).stdout
)
print(
    "NOTE: `persistence.size: 10Gi` in values.yaml is NOT what this shows - OrbStack's\n"
    "local-path provisioner does not enforce the requested capacity. Check the PVC spec,\n"
    "not df, if you need to verify that setting."
)

## 5 — Plant the probe table, and confirm it replicated

Created **through Pgpool** (a write, so it lands on the primary) in the `polaris` database as the
`polaris` user — the same database and role the reads use. A hand-run created it as `postgres` in
the `postgres` database and read it as `polaris` in `polaris`, producing twelve identical
`relation "rw_probe" does not exist` errors and no result.

Created *before* the pause, so the experiment measures **row visibility**, not DDL visibility.

In [ ]:
pool_sql(
    f"CREATE TABLE IF NOT EXISTS {PROBE_TABLE} "
    "(id serial PRIMARY KEY, t timestamptz DEFAULT now());",
    fetch=False,
)
time.sleep(2)

seen = {i: kexec(i, f"SELECT to_regclass('{PROBE_TABLE}') IS NOT NULL;") for i in STANDBYS}
print({f"pg-{i}": v for i, v in seen.items()})
assert all(v == "t" for v in seen.values()), (
    f"probe table has not replicated to every standby: {seen}. Do not pause yet."
)
print(f"\n'{PROBE_TABLE}' exists on every standby - safe to pause")

## 6 — Pause replay on **every** standby

`pg_wal_replay_pause()` is a *request*; replay stops at the next recovery pause point, so the
state is checked rather than assumed. One standby left running serves fresh reads and masks the
whole effect, so this asserts on all of them.

In [ ]:
for i in STANDBYS:
    kexec(i, "SELECT pg_wal_replay_pause();")
time.sleep(1)

states = {i: kexec(i, "SELECT pg_get_wal_replay_pause_state();") for i in STANDBYS}
print({f"pg-{i}": v for i, v in states.items()})
assert all(v == "paused" for v in states.values()), (
    f"not every standby is paused: {states}. 'pause requested' means replay has not "
    "stopped yet - wait a moment and re-run this cell."
)
print("\nall standbys paused - the clock is running, finish and resume promptly")

## 7 — Does a pooled read miss a committed write?

One write through Pgpool, ground truth straight off the primary, then `N_SESSIONS` brand-new
pooled sessions running a **plain** `count(*)` — no system functions, nothing that could change
how Pgpool classifies the statement.

Routing comes from Pgpool's `select_cnt` counters, diffed around the loop.

In [ ]:
before_rows = int(kexec(PRIMARY, f"SELECT count(*) FROM {PROBE_TABLE};"))
pool_sql(f"INSERT INTO {PROBE_TABLE} DEFAULT VALUES;", fetch=False)
truth = int(kexec(PRIMARY, f"SELECT count(*) FROM {PROBE_TABLE};"))
print(f"rows on the primary: {before_rows} -> {truth}   (ground truth = {truth})\n")

cnt_before = select_counts()
observed = []
for n in range(1, N_SESSIONS + 1):
    ((n_rows,),) = pool_sql(f"SELECT count(*) FROM {PROBE_TABLE};")
    observed.append({"session": n, "rows_seen": n_rows, "stale": n_rows < truth})
cnt_after = select_counts()

routing = pd.DataFrame(
    [
        {"hostname": hn, "selects_served": cnt_after[hn] - cnt_before.get(hn, 0)}
        for hn in cnt_after
    ]
)
probe = pd.DataFrame(observed)

print("routing, from Pgpool's own counters:")
display(routing)
print(f"total selects attributed: {int(routing.selects_served.sum())} (expected ~{N_SESSIONS})")
display(probe)

### 7b — Instrument check

The same 12 sessions with run 1's query, which embeds `pg_is_in_recovery()`. If §7's counters
show standbys serving reads while this reports `primary` every time, the detector was the
confound and run 1's REFUTED verdict is explained.

In [ ]:
cnt_before_i = select_counts()
instrumented = []
for n in range(1, N_SESSIONS + 1):
    (in_recovery, n_rows, addr), = pool_sql(
        "SELECT pg_is_in_recovery(), count(*), inet_server_addr()::text "
        f"FROM {PROBE_TABLE};"
    )
    instrumented.append(
        {"session": n, "reports": "standby" if in_recovery else "primary",
         "addr": addr, "rows_seen": n_rows}
    )
cnt_after_i = select_counts()

instr = pd.DataFrame(instrumented)
instr_routing = {hn: cnt_after_i[hn] - cnt_before_i.get(hn, 0) for hn in cnt_after_i}
display(instr)
print("counters for the instrumented query:", instr_routing)
print(f"the query itself reported standby {int((instr.reports == 'standby').sum())}/{N_SESSIONS}")

## 8 — The #15 ladder, while replication is stale

The symptom, not the mechanism: `N_RUNGS` brand-new catalogs, each immediately followed by the
first namespace created in it. That is exactly the shape that produced 3-of-3 500s in #15.

In [ ]:
CREATED = []


def ladder(n_rungs, tag):
    """Create n_rungs brand-new catalogs, each followed at once by its first
    namespace. Returns a row per rung with both status codes."""
    tok = root_token()
    out = []
    for i in range(n_rungs):
        cat = f"rw{int(time.time())}{tag}{i}"
        rc = create_catalog(cat, token=tok)
        CREATED.append(cat)
        rn = create_namespace(catalog=cat, ns="probe_ns", token=tok)
        out.append(
            {
                "phase": tag,
                "rung": i,
                "catalog": cat,
                "catalog_status": rc.status_code,
                "namespace_status": rn.status_code,
            }
        )
    return out


paused_run = pd.DataFrame(ladder(N_RUNGS, "paused"))
display(paused_run)
paused_500 = int((paused_run.namespace_status == 500).sum())
print(f"namespace creates returning 500 while STALE: {paused_500}/{N_RUNGS}")

## 9 — Resume replication

In [ ]:
resume_all()
time.sleep(5)

print("\nreplication after resume (client | state | replay_lag):")
print(
    kexec(
        PRIMARY,
        "SELECT client_addr || ' | ' || state || ' | ' || "
        "coalesce(replay_lag::text, '0') FROM pg_stat_replication;",
    )
)

## 10 — The control: the same ladder against healthy replication

Without this, a 500 rate while paused says nothing — #15's 500s were observed on a cluster nobody
had paused. The comparison is the finding.

In [ ]:
healthy_run = pd.DataFrame(ladder(N_RUNGS, "healthy"))
display(healthy_run)
healthy_500 = int((healthy_run.namespace_status == 500).sum())
print(f"namespace creates returning 500 while HEALTHY: {healthy_500}/{N_RUNGS}")

## 11 — Verdict

In [ ]:
served = {r.hostname: int(r.selects_served) for r in routing.itertuples()}
primary_host = nodes.loc[nodes.role.astype(str).str.lower().str.contains("primary"), "hostname"]
primary_host = primary_host.iloc[0] if len(primary_host) else None
standby_selects = sum(v for k, v in served.items() if k != primary_host)
n_stale = int(probe.stale.sum())

print("=== mechanism (does a pooled read miss a committed write?) ===")
if standby_selects == 0:
    mech = (
        f"NO BALANCING - Pgpool served all {N_SESSIONS} reads from the primary "
        f"({primary_host}). Check lb_weight and status in the 2b table: a zero weight or a "
        "down backend explains this without any read-after-write behaviour existing."
    )
elif n_stale > 0:
    mech = (
        f"CONFIRMED - {standby_selects}/{N_SESSIONS} reads were served by a standby and "
        f"{n_stale} of them returned a stale count against a ground truth of {truth}. A write "
        "on one pooled connection is invisible to a read on another."
    )
else:
    mech = (
        f"INCONCLUSIVE - {standby_selects}/{N_SESSIONS} reads went to a standby but none read "
        f"stale (all saw {truth}) even with replay paused. Re-check the pause state."
    )
print(mech)

print("\n=== symptom (does the #15 ladder 500?) ===")
if paused_500 > healthy_500:
    symp = (
        f"CORRELATED - {paused_500}/{N_RUNGS} namespace creates 500'd while replication was "
        f"stale vs {healthy_500}/{N_RUNGS} healthy. Staleness makes the #15 500s more likely, "
        "which is the first direct evidence for the read-after-write reading."
    )
elif paused_500 == healthy_500 == 0:
    symp = (
        f"NOT REPRODUCED - 0/{N_RUNGS} 500s in either phase. The ladder did not provoke #15 "
        "today, so this run says nothing about the symptom either way. #15's own evidence was "
        "3-of-3 on a differently-seeded cluster; check whether the fixture differs."
    )
elif paused_500 == healthy_500:
    symp = (
        f"UNCORRELATED - {paused_500}/{N_RUNGS} 500s in BOTH phases. The 500s do not depend on "
        "replication lag, which points at Polaris's own resolver rather than the database."
    )
else:
    symp = (
        f"INVERTED - {paused_500} 500s while stale vs {healthy_500} healthy. Not a lag effect; "
        "treat the ladder as noisy and re-run with more rungs."
    )
print(symp)

print("\n=== what follows ===")
if standby_selects > 0 and n_stale > 0 and paused_500 > healthy_500:
    print("Set database_redirect_preference_list = 'polaris:primary' in pgpool.configuration")
    print("(the 'primary' KEYWORD, not a node id - an id pins reads to a demoted standby after")
    print("a failover), then re-run this notebook to measure how much of the 500 rate it removes.")
elif standby_selects > 0 and n_stale > 0:
    print("The mechanism is real but did not move the 500 rate here. Pinning reads to the primary")
    print("is still correct on its own merits; it just is not established as #15's cause.")
else:
    print("No read-after-write effect demonstrated. #15 stays with Polaris's resolver, and since")
    print("Polaris is not to be changed the remedy is a client-side retry. The Toxiproxy")
    print("failover harness loses its urgency: see local-k8s HANDOFF-toxiproxy-failover.")

## 12 — Tear down

In [ ]:
for cat in CREATED:
    try:
        teardown_catalog(cat)
        print(f"torn down {cat}")
    except Exception as e:  # noqa: BLE001
        print(f"WARNING: {cat} not fully removed: {e}")

pool_sql(f"DROP TABLE IF EXISTS {PROBE_TABLE};", fetch=False)
print(f"dropped {PROBE_TABLE}")

print("\nfinal replay state:")
for i in range(REPLICAS):
    try:
        print(f"  pg-{i}: {kexec(i, 'SELECT pg_get_wal_replay_pause_state();') or 'primary'}")
    except Exception:  # noqa: BLE001
        print(f"  pg-{i}: primary")

In [ ]:
import gc

for _n in ("probe", "routing", "instr", "roles", "nodes", "paused_run", "healthy_run"):
    globals().pop(_n, None)
gc.collect()

# No cursors, engines or pools are held: pool_sql opens and closes one connection
# per call by design, and kexec speaks to kubectl, not to a driver.
print("teardown complete - nothing left open")

## How to read it

The two verdicts are independent and both matter.

**Mechanism** answers whether stale reads are possible here at all. **Symptom** answers whether
they cause #15. A CONFIRMED mechanism with an UNCORRELATED symptom is a real and useful result:
it means pinning reads to the primary is worth doing but will not fix the 500s.

**A correlated result still is not the whole diagnosis.** #15's signature is 3 of 3 brand-new
catalogs failing — deterministic, where probabilistic lag would be intermittent. And the standbys
were measured `streaming` at near-zero lag under normal load, so the natural staleness window is
small. Expect staleness to explain some of the rate, not all of it.

**The denominator is still unavailable from the log pipeline.** Per `local-k8s` #15, a
*successful* namespace create leaves no individual access-log record, so the true 500 rate has to
come from a driver recording its own statuses — which is what §8 and §10 do, for these rungs only.

## Related

- `local-k8s/postgresql/HANDOFF-toxiproxy-failover-2026-09-17.md` — the primary-unavailable
  harness, deferred behind this question
- `local-k8s/.memory/active-issues.md` #15 (the 500s), #9 (plaintext JDBC password)
- `local-k8s/postgresql/values.yaml` — `pgpool.configuration` is where the remedy goes